In [9]:
import boto3
import pandas as pd
from io import BytesIO

In [14]:
BUCKET_NAME = "kltn-customer-analytics-2026"
FILE_KEY = "bronze/online_retail_II.xlsx"

s3 = boto3.client("s3")

response = s3.get_object(
    Bucket=BUCKET_NAME,
    Key=FILE_KEY
)

excel_data = BytesIO(response["Body"].read())

# Kiểm tra các sheet
xls = pd.ExcelFile(excel_data, engine="openpyxl")

print("Danh sách sheet:", xls.sheet_names)

Danh sách sheet: ['Year 2009-2010', 'Year 2010-2011']


In [15]:
sheets = pd.read_excel(
    xls,
    sheet_name=None
)

df = pd.concat(
    sheets.values(),
    ignore_index=True
)

print("Số dòng:", df.shape[0])
print("Số cột:", df.shape[1])

df.head()

Số dòng: 1067371
Số cột: 8


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [16]:
print("Danh sách cột:")
print(df.columns.tolist())

print("\nThông tin dữ liệu:")
df.info()

Danh sách cột:
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']

Thông tin dữ liệu:
<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[us]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 65.1+ MB


In [17]:
df["InvoiceDate"] = pd.to_datetime(
    df["InvoiceDate"],
    errors="coerce"
)

print("Ngày đầu tiên:", df["InvoiceDate"].min())
print("Ngày cuối cùng:", df["InvoiceDate"].max())

Ngày đầu tiên: 2009-12-01 07:45:00
Ngày cuối cùng: 2011-12-09 12:50:00


In [19]:
df.shape

(1067371, 8)

In [20]:
print("Kích thước dữ liệu:", df.shape)
print("Danh sách cột:", df.columns.tolist())

df["InvoiceDate"] = pd.to_datetime(
    df["InvoiceDate"],
    errors="coerce"
)

print("Ngày giao dịch đầu:", df["InvoiceDate"].min())
print("Ngày giao dịch cuối:", df["InvoiceDate"].max())
print("Số ngày không hợp lệ:", df["InvoiceDate"].isna().sum())

Kích thước dữ liệu: (1067371, 8)
Danh sách cột: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']
Ngày giao dịch đầu: 2009-12-01 07:45:00
Ngày giao dịch cuối: 2011-12-09 12:50:00
Số ngày không hợp lệ: 0


In [21]:
column_info = pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str).values,
    "Non-null": df.notna().sum().values,
    "Missing": df.isna().sum().values
})

column_info

,Column,Data Type,Non-null,Missing
0,Invoice,object,1067371,0
1,StockCode,object,1067371,0
2,Description,object,1062989,4382
3,Quantity,int64,1067371,0
4,InvoiceDate,datetime64[us],1067371,0
5,Price,float64,1067371,0
6,Customer ID,float64,824364,243007
7,Country,str,1067371,0


In [22]:
from pandas.core.array_algos.quantile import quantile_with_mask
from pandas.core.array_algos import quantile
from pandas.core.array_algos import quantile
invoice = df["Invoice"].astype("string").str.strip()

quality_report = {
    "Total Rows": len(df),
    "Missing Customer ID": df["Customer ID"].isna().sum(),
    "Missing Description": df["Description"].isna().sum(),
    "Duplicate Rows": df.duplicated().sum(),
    "Cancelled Invoice Rows": invoice.str.upper().str.startswith("C", na=False).sum(),
    "Quantity <= 0": (df["Quantity"] <= 0).sum(),
    "Price <= 0": (df["Price"] <= 0).sum(),
    "Invalid InvoiceDate": df["InvoiceDate"].isna().sum(),
    "Unique Customers": df["Customer ID"].nunique()
}

quality_df = pd.DataFrame(
    quality_report.items(),
    columns=["Indicator", "Count"]
)

quality_df

,Indicator,Count
0,Total Rows,1067371
1,Missing Customer ID,243007
2,Missing Description,4382
3,Duplicate Rows,34335
4,Cancelled Invoice Rows,19494
5,Quantity <= 0,22950
6,Price <= 0,6207
7,Invalid InvoiceDate,0
8,Unique Customers,5942


In [23]:
df[["Quantity", "Price"]].describe(
        percentiles=[0.01, 0.25, 0.50, 0.75, 0.95, 0.99]
    )

,Quantity,Price
count,1.067371e+06,1.067371e+06
mean,9.938898e+00,4.649388e+00
std,1.727058e+02,1.235531e+02
min,-8.099500e+04,-5.359436e+04
1%,-3.000000e+00,2.100000e-01
25%,1.000000e+00,1.250000e+00
50%,3.000000e+00,2.100000e+00
75%,1.000000e+01,4.150000e+00
95%,3.000000e+01,9.950000e+00
99%,1.000000e+02,1.800000e+01


In [24]:
invoice = df["Invoice"].astype("string").str.strip()
cancelled = invoice.str.upper().str.startswith("C", na=False)

print("KIỂM TRA GIAO DỊCH")

print(
    "Hóa đơn hủy có Quantity <= 0:",
    (cancelled & (df["Quantity"] <= 0)).sum()
)

print(
    "Quantity <= 0 nhưng không thuộc hóa đơn hủy:",
    ((df["Quantity"] <= 0) & ~cancelled).sum()
)

print(
    "Price < 0:",
    (df["Price"] < 0).sum()
)

print(
    "Price == 0:",
    (df["Price"] == 0).sum()
)

print("\n MẪU CÁC DÒNG PRICE ÂM")
df.loc[df["Price"] < 0].head(10)

print("\nMẪU CÁC DÒNG QUANTITY ÂM, KHÔNG HỦY")

df.loc[(df["Quantity"] < 0) & ~cancelled].head(10)

print("\nMẪU CÁC DÒNG TRÙNG")
df.loc[df.duplicated(keep=False)].head(10)

KIỂM TRA GIAO DỊCH
Hóa đơn hủy có Quantity <= 0: 19493
Quantity <= 0 nhưng không thuộc hóa đơn hủy: 3457
Price < 0: 5
Price == 0: 6202

 MẪU CÁC DÒNG PRICE ÂM

MẪU CÁC DÒNG QUANTITY ÂM, KHÔNG HỦY

MẪU CÁC DÒNG TRÙNG


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
362,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
363,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
367,489517,22319,HAIRCLIPS FORTIES FABRIC ASSORTED,12,2009-12-01 11:34:00,0.65,16329.0,United Kingdom
368,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,16329.0,United Kingdom
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
379,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom
383,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,16329.0,United Kingdom
384,489517,22319,HAIRCLIPS FORTIES FABRIC ASSORTED,12,2009-12-01 11:34:00,0.65,16329.0,United Kingdom
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
